# Master Benchmark Evaluation Pipeline

## Overview
This notebook performs cross-configuration consolidation and comprehensive comparative analysis across all trained models:
- **SFT Baseline**: Supervised fine-tuning reference model (`T5-base` trained on `s-nlp/paradetox`).
- **Curriculum PPO Configurations (A through G - v1)**: Multi-seed curriculum RL pipeline evaluated across Phase 1 (Easy), Phase 2 (Medium), and Phase 3 (Full Refinement).
- **Flat PPO Baseline (Config C - v1)**: Compute-matched flat reinforcement learning baseline across seeds 42, 123, and 777.

## Key Evaluation Procedures
1. **Per-Sample Aggregation**: Reads and pools all inference generation outputs (`final_results`) on toxic-only test splits.
2. **Dual-Classifier STA Audit & Agreement**: Assesses inter-annotator agreement and Cohen's Kappa ($\kappa$) between the primary reward classifier (`unitary/unbiased-toxic-roberta`) and the independent evaluation classifier (`s-nlp/roberta_toxicity_classifier`) under calibration threshold $\tau_{\text{out}} = 0.3$.
3. **Perplexity (PPL) Verification**: Uses `gpt2` with prepended BOS tokens to robustly estimate language model fluency without short-sentence artifacts.
4. **Deletion Bias Analysis**: Compares length ratios ($|y|/|x| < 0.70$) against the original ParaDetox training corpus to assess whether length reductions are inherited from SFT or driven by RL reward objectives.

In [ ]:

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import cohen_kappa_score
import warnings
warnings.filterwarnings('ignore')

ROOT = Path("/content/drive/MyDrive/Data_Khoaluan")
OUTPUT_DIR = ROOT / "Master_Evaluation_Outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_ROOTS = {
    "A": ROOT / "ppo_config_a_v1",
    "B": ROOT / "ppo_config_b_v1",
    "C": ROOT / "ppo_config_c_v1",
    "D": ROOT / "ppo_config_d_v1",
    "E": ROOT / "ppo_config_e_v1",
    "F": ROOT / "ppo_config_f_v1",
    "G": ROOT / "ppo_config_g_v1",
    "Flat_PPO": ROOT / "flat_ppo_config_c_v1"
}

SEEDS = [42, 123, 777]
TAU_OUT = 0.3
SEVERE_LEN_CUTOFF = 0.7


Mounted at /content/drive


In [ ]:

def load_all_data():
    frames = []

    # 1. SFT Baseline
    sft_path = CONFIG_ROOTS["A"] / "final_results" / "baseline_per_sample.csv"
    if sft_path.exists():
        df = pd.read_csv(sft_path)
        df['config'] = 'SFT'
        df['seed'] = 42 # SFT only has 1 run but we map it to seeds for aggregation if needed, or keep 0
        df['phase'] = 'Baseline (SFT)'

        ppl_path = CONFIG_ROOTS["A"] / "final_results" / "ppl_full.csv"
        if ppl_path.exists():
            ppl_df = pd.read_csv(ppl_path)
            if 'ppl' in ppl_df.columns: df['ppl'] = ppl_df['ppl'].values[:len(df)]

        frames.append(df)

    # 2. Config A-G (Phase 3)
    for cfg, root_dir in CONFIG_ROOTS.items():
        if root_dir is None: continue
        for seed in SEEDS:
            p = Path(root_dir) / "final_results" / f"per_sample_seed{seed}_phase3_refinement.csv"
            if p.exists():
                df = pd.read_csv(p)
                df['config'] = cfg
                df['seed'] = seed
                df['phase'] = 'Phase 3 (Full)'

                # PPL
                ppl_path = Path(root_dir) / "final_results" / "ppl_full.csv"
                if ppl_path.exists():
                    ppl_df = pd.read_csv(ppl_path)
                    ppl_df_seed = ppl_df[ppl_df['seed'] == seed] if 'seed' in ppl_df.columns else ppl_df
                    if 'ppl' in ppl_df_seed.columns:
                        df['ppl'] = ppl_df_seed['ppl'].values[:len(df)]

                frames.append(df)

            # Load phase1_easy and phase2_balance for all configs
            for phase_file, phase_name in [('phase1_easy', 'Phase 1 (Easy)'), ('phase2_balance', 'Phase 2 (+Med)')]:
                p_phase = Path(root_dir) / "final_results" / f"per_sample_seed{seed}_{phase_file}.csv"
                if p_phase.exists():
                    df_p = pd.read_csv(p_phase)
                    df_p['config'] = cfg
                    df_p['seed'] = seed
                    df_p['phase'] = phase_name
                    frames.append(df_p)

    # 3. Flat PPO
    for seed in SEEDS:
        p = ROOT / "flat_ppo_config_c_v1" / "final_results" / f"per_sample_seed{seed}_phase_flat.csv"
        if p.exists():
            df = pd.read_csv(p)
            df['config'] = 'Flat_PPO'
            df['seed'] = seed
            df['phase'] = 'Phase 3 (Full)'
            frames.append(df)

    if not frames:
        raise ValueError("No data found!")

    all_df = pd.concat(frames, ignore_index=True)
    if 'X' in all_df.columns: all_df = all_df.rename(columns={'X': 'input', 'Y': 'output'})
    if 'sta' in all_df.columns: all_df['STA'] = all_df['sta']
    if 'sta_indep' in all_df.columns: all_df['STA (indep.)'] = all_df['sta_indep']
    if 'sim' in all_df.columns: all_df['Sim'] = all_df['sim']
    if 'joint' in all_df.columns: all_df['Joint'] = all_df['joint']
    if 'flu' in all_df.columns: all_df['Flu'] = all_df['flu']
    if 'ppl' in all_df.columns: all_df['ppl'] = all_df['ppl']

    # Compute length columns
    all_df['x_len'] = all_df['input'].astype(str).apply(lambda x: len(x.split()))
    all_df['y_len'] = all_df['output'].astype(str).apply(lambda x: len(x.split()))
    all_df['Severe_Len_Red'] = (all_df['y_len'] / all_df['x_len']) < SEVERE_LEN_CUTOFF
    all_df['Unchanged'] = (all_df['input'].astype(str) == all_df['output'].astype(str))

    return all_df

df_all = load_all_data()
print("Loaded data successfully. Initial size:", len(df_all))

# TOXIC-ONLY FILTER
# Keep only samples with label 'toxic'
if 'label' in df_all.columns:
    original_len = len(df_all)
    # Filter rows with 'toxic' (case-insensitive)
    df_all = df_all[df_all['label'].astype(str).str.lower().str.strip() == 'toxic'].reset_index(drop=True)
    print(f"Filtered Toxic-Only (label is 'toxic'). Rows reduced from {original_len} to {len(df_all)}")
else:
    print("Label column not found for Toxic-Only filtering. Using full dataset.")


Loaded data successfully. Initial size: 87904
Filtered Toxic-Only (label is 'toxic'). Rows reduced from 87904 to 77050


In [ ]:

import torch
import math
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

if 'ppl' not in df_all.columns or df_all['ppl'].isna().all():
    print("Starting PPL (Perplexity) computation for all output sentences...")
    # Use gpt2 for fast PPL calculation
    ppl_model_id = "gpt2"

    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        ppl_tokenizer = AutoTokenizer.from_pretrained(ppl_model_id)
        ppl_model = AutoModelForCausalLM.from_pretrained(ppl_model_id).to(device)
        ppl_model.eval()

        def calculate_ppl(text):
            if not isinstance(text, str) or len(text.strip()) == 0:
                return float('inf')

            # Mitigate high PPL spikes on short sentences:
            # Prepend BOS token (<|endoftext|>) to prompt GPT-2
            full_text = ppl_tokenizer.eos_token + text

            encodings = ppl_tokenizer(full_text, return_tensors='pt')
            seq_len = encodings.input_ids.size(1)

            if seq_len < 2: return float('inf')

            with torch.no_grad():
                inputs = encodings.input_ids.to(device)
                # Batch size 1 (sentence by sentence), GPT-2 auto-shifts labels
                outputs = ppl_model(inputs, labels=inputs)
                loss = outputs.loss
            return torch.exp(loss).item()

        tqdm.pandas(desc="Computing PPL")
        df_all['ppl'] = df_all['output'].progress_apply(calculate_ppl)
        print("PPL computation completed!")
    except Exception as e:
        print(f"Error loading GPT-2 for PPL computation: {e}")
else:
    print("Column 'ppl' already exists, skipping re-computation.")


Starting PPL (Perplexity) computation for all output sentences...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Computing PPL:   0%|          | 0/77050 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


PPL computation completed!


In [ ]:

print("="*80)
print("PARADETOX LENGTH STATISTICS (DID SFT INHERIT DELETION BIAS?)")
print("="*80)
try:
    import datasets
    from datasets import load_dataset
    print("Downloading ParaDetox from HuggingFace...")
    dataset = load_dataset("s-nlp/paradetox")
    train_df = dataset['train'].to_pandas()

    col_x = 'en_toxic_comment'
    col_y = 'en_neutral_comment'

    train_df['x_len'] = train_df[col_x].astype(str).apply(lambda x: len(x.split()))
    train_df['y_len'] = train_df[col_y].astype(str).apply(lambda x: len(x.split()))
    train_df['len_ratio'] = train_df['y_len'] / train_df['x_len']

    ratio_cut = (train_df['len_ratio'] < 0.7).mean() * 100
    avg_ratio = train_df['len_ratio'].mean()

    print(f"Ratio of ParaDetox pairs with |y| < 0.7|x| : {ratio_cut:.2f}%")
    print(f"Average |y|/|x| ratio                     : {avg_ratio:.4f}")
except Exception as e:
    print("Error loading or processing ParaDetox from Hugging Face:", e)


PARADETOX LENGTH STATISTICS (DID SFT INHERIT DELETION BIAS?)


README.md:   0%|          | 0.00/7.59k [00:00<?, ?B/s]

train.tsv:   0%|          | 0.00/2.04M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19744 [00:00<?, ? examples/s]

Ratio of ParaDetox pairs with |y| < 0.7|x| : 29.21%
Average |y|/|x| ratio                     : 0.7797


In [ ]:
from sklearn.metrics import cohen_kappa_score
import numpy as np
import pandas as pd

print("\n" + "="*80)
print(f"3. AGREEMENT RATE AND KAPPA (κ) (THRESHOLD TAU_OUT = {TAU_OUT})")
print("="*80)

print(f"Primary scorer (STA)     : unitary/unbiased-toxic-roberta")
print(f"Independent scorer (indep) : s-nlp/roberta_toxicity_classifier")
print("-" * 80)
print(f"{'Configuration':<15} | {'Phase':<20} | {'n':<6} | {'Agreement (%)':<15} | {'Cohen Kappa (κ)'}")
print("-" * 80)

config_list = df_all['config'].dropna().unique()

for cfg in config_list:
    cfg_data = df_all[df_all['config'] == cfg]

    if 'Phase 3 (Full)' in cfg_data['phase'].values:
        df_eval = cfg_data[cfg_data['phase'] == 'Phase 3 (Full)']
        ph_label = 'Phase 3 (Full)'
    else:
        df_eval = cfg_data
        ph_label = 'All (Original)'

    # Ensure both raw score columns exist
    col_main = 'tox' if 'tox' in df_eval.columns else 'STA'
    col_indep = 'tox_indep' if 'tox_indep' in df_eval.columns else 'STA (indep.)'

    if col_main in df_eval.columns and col_indep in df_eval.columns:
        df_eval = df_eval.dropna(subset=[col_main, col_indep])

    if len(df_eval) > 0:
        # If column contains raw scores (e.g. 0.15, 0.85), threshold with < TAU_OUT
        # If column is already binarized (0/1), preserve

        main_vals = pd.to_numeric(df_eval[col_main], errors='coerce')
        indep_vals = pd.to_numeric(df_eval[col_indep], errors='coerce')

        # Dynamic detection: preserve if binarized, threshold with TAU_OUT if continuous
        if main_vals.max() <= 1.0 and len(main_vals.unique()) > 2:
            sta1 = (main_vals < TAU_OUT).astype(int)
            sta2 = (indep_vals < TAU_OUT).astype(int)
        else:
            # Pre-binarized STA column
            sta1 = main_vals.astype(int)
            sta2 = indep_vals.astype(int)

        agreement = (sta1 == sta2).mean() * 100
        kappa = cohen_kappa_score(sta1, sta2)

        print(f"{cfg:<15} | {ph_label:<20} | {len(df_eval):<6} | {agreement:<15.2f} | {kappa:.4f}")
    else:
        print(f"{cfg:<15} | {ph_label:<20} | {'0':<6} | {'N/A':<15} | N/A")

print("-" * 80)
print("-> Note: Safe threshold TAU_OUT = 0.3 applied on raw Output scores.")
print("   Report Kappa (κ) as primary metric. κ >= 0.6 indicates Substantial agreement.")



3. AGREEMENT RATE AND KAPPA (κ) (THRESHOLD TAU_OUT = 0.3)
Primary scorer (STA)     : unitary/unbiased-toxic-roberta
Independent scorer (indep) : s-nlp/roberta_toxicity_classifier
--------------------------------------------------------------------------------
Configuration        | Phase                | n      | Agreement (%)  | Cohen Kappa (κ)
--------------------------------------------------------------------------------
SFT             | All (Original)            | 1150   | 94.87           | 0.6505
A               | Phase 3 (Full)       | 3450   | 97.39           | 0.5580
B               | Phase 3 (Full)       | 3450   | 97.28           | 0.5468
C               | Phase 3 (Full)       | 3450   | 96.32           | 0.6622
D               | Phase 3 (Full)       | 3450   | 98.09           | 0.5503
E               | Phase 3 (Full)       | 3450   | 95.91           | 0.6763
F               | Phase 3 (Full)       | 3450   | 96.20           | 0.6730
G               | Phase 3 (Full)       |